# An M-Dwarf Flares Example

In this notebook we run a simulation of the built-in m-dwarf flares model from Mendoza et al. (2022), with free parameters for amplitude and full width half max of the flare.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

from lightcurvelynx.graph_state import GraphState
from lightcurvelynx.math_nodes.given_sampler import GivenValueList
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.models.mdwarf_flare_model import MDwarfFlareModel
from lightcurvelynx.obstable.lsst_obstable import LSSTObsTable
from lightcurvelynx.simulate import compute_single_noise_free_lightcurve, simulate_lightcurves
from lightcurvelynx.survey_info import SurveyInfo
from lightcurvelynx.utils.plotting import plot_lightcurves

## Fake Survey Data

For the purposes of this notebook, we create a very basic survey using the LSST instrument that looks at one region of the sky. It revisits the same patch `samples_per_day` times a night in alternating filters ("g", "r", "i", "z", and "u").

For more realistic surveys, see the examples for [Rubin OpSim](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/opsim_notebook.html), [Rubin CCDVisit Table](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/pre_executed/rubin_ccdvisit.html), or [Custom Surveys](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/custom_survey.html).

In [ ]:
filters = ["g", "r", "i", "z", "u"]
num_days = 4
samples_per_day = 100  # 20 samples in each filter
num_samples = num_days * samples_per_day

times = np.arange(num_samples) / samples_per_day + 60676.0  # MJD starting at 60676.0
fake_data = {
    "expMidptMJD": times,  # MJD
    "ra": np.full(num_samples, 150.0),  # Degrees
    "dec": np.full(num_samples, -30.0),  # Degrees
    "band": [filters[i % len(filters)] for i in range(num_samples)],
    "exptime": np.full(num_samples, 30.0),  # Seconds
    # Noise information is random from distributions around DP2 values.
    "pixelScale": np.full(num_samples, 0.2),  # Arcseconds per pixel
    "seeing": np.random.uniform(low=0.84, high=0.88, size=num_samples),  # arcseconds
    "skyBg": np.random.uniform(low=1420.0, high=1430.0, size=num_samples),  # ADU
    "zeroPoint": np.random.normal(
        loc=30.0, scale=0.5, size=num_samples
    ),  # magnitudes to produce 1 count (ADU)
}
obs_table = LSSTObsTable(fake_data)
obs_table.head()

Since we are using a standard survey type (LSST), we can just pass the name to the `SurveyInfo` info constructor and have it load the default passbands and noise model.

In [ ]:
survey_info = SurveyInfo(obstable=obs_table, survey_name="LSST")

## Create Parameter Distributions

With LightCurveLynx model parameters may be both fixed numerical values or to be drawn from random distributions. Here we create samplers for the various m-dwarf flare parameters, including: `star_temp`, `star_radius`, `flare_fwhm`, and `flare_amplitude`. 

The m-dwarf flares model has a built-in prior model for these parameters if they are not specified.

In [ ]:
# Star temperature is fixed at 4000 K.
star_temp = 4000

# Star radius is drawn from a normal distribution centered at 0.09 solar radii.
star_radius_sampler = NumpyRandomFunc(
    "normal",
    loc=0.09 * u.R_sun.to(u.cm),
    scale=0.01 * u.R_sun.to(u.cm),
)

# Flare FWHM is drawn from a uniform distribution [0.05, 0.15] in the same units as t0.
flare_fwhm_sampler = NumpyRandomFunc("uniform", low=0.05, high=0.15)

# Flare amplitude is drawn from a normal distribution centered at 0.05.
flare_amplitude_sampler = NumpyRandomFunc("normal", loc=0.05, scale=0.01)

# t0 is fixed at the midpoint of our fake OpSim coverage, so we can see the flare clearly.
t0 = (times.min() + times.max()) / 2

## Create the Model

Now that we have the parameter setters, we can define the m-dwarf flare mode. We set the physical model's parameters from the setters defined in the previous cell and set the positional parameters to constants so they appear in each image.

In [ ]:
model = MDwarfFlareModel(
    star_temp=star_temp,
    star_radius=star_radius_sampler,
    flare_fwhm=flare_fwhm_sampler,
    flare_amplitude=flare_amplitude_sampler,
    t0=t0,
    ra=150.0,  # We always see the object
    dec=-30.0,  # We always see the object
    distance=100.0,  # Fixed distance in parsecs
    node_label="model",
)

We can generate samples to see the values of the parameters.

In [ ]:
state = model.sample_parameters(num_samples=10)
for key, value in state["model"].items():
    print(f"{key}: {value}")

## Run Simulations

Now let's simulate a few light curves!

In [ ]:
# Fix the random seed to make it reproducible.
rng = np.random.default_rng(42)

df = simulate_lightcurves(
    model=model,
    num_samples=3,
    survey_info=survey_info,
    rng=rng,
)

Then we plot the simulated light curves in magnitude space.

In [ ]:
def plot_all_results(df):
    """A simple helper function to plot the mdwarf flares in the results

    Parameters
    ----------
    df : nested_pandas.NestedFrame
        The results data frame.
    """
    for idx in range(len(df)):
        # Extract the row for this object.
        row = df.iloc[idx]
        lc = row["lightcurve"]
        params = row["params"]

        if lc is None or len(lc) == 0:
            print(f"Skipping empty result at index={idx}")
            continue

        # Extract the parameters used during the simulation of this object. Use that to
        # compute the noise-free light curve for this object, which we will plot alongside
        # the simulated light curve.
        noise_free_lcs = compute_single_noise_free_lightcurve(
            model,
            GraphState.from_dict(params),
            survey_info.passbands,
            rest_frame_phase_min=-num_days / 2,
            rest_frame_phase_max=num_days / 2,
            rest_frame_phase_step=0.01,
        )

        plot_lightcurves(
            fluxes=lc["flux"],
            times=lc["mjd"],
            fluxerrs=lc["fluxerr"],
            filters=lc["filter"],
            underlying_model=noise_free_lcs,
            plot_magnitudes=True,
        )

        # Plot a vertical line at t0, which is the reference time for the light curve.
        plt.axvline(x=params["model.t0"], color="k", linestyle="--")
        plt.title(f"Result {idx}")
        plt.show()


plot_all_results(df)

## Filtering rare occurrences

One of the challenges with mdwarf flares is that they only occur for short durations (days). LightCurveLynx's builtin filtering will evaluate the model at every time point the object's position in the sky (RA, Dec) is observed. This means we could get a large number of useless observations. Worse, none of these observations may include the flare itself.

### Example

To see this, let's create a new data set with observations of three different flares at different locations:
1) Observations at (0, 0) that includes just the time around the flare.
2) Observations at (180, -30) that include the flare and a lot of extra observations before and after.
3) Observations at (90, 30) that do not include the flare itself.

To make things simple, we have all three flares occur at the same time `t0`=60776.0

In [ ]:
t0 = 60776.0

# The first set of observations contains just observations around the time of the flare.
times1 = t0 + np.arange(-2, 2, 0.5)
ra1 = np.full(len(times1), 0.0)
dec1 = np.full(len(times1), 0.0)

# The second set of observations contain a lot of times before and after the flare.
times2 = t0 + np.arange(-50, 50, 0.5)
ra2 = np.full(len(times2), 180.0)
dec2 = np.full(len(times2), -30.0)

# The third set of observations contain only times before the flare.
times3 = t0 + np.arange(-50, -5, 0.5)
ra3 = np.full(len(times3), 90.0)
dec3 = np.full(len(times3), 30.0)

all_times = np.concat((times1, times2, times3))
num_samples = len(all_times)

fake_data = {
    "expMidptMJD": all_times,  # MJD
    "ra": np.concat((ra1, ra2, ra3)),  # Degrees
    "dec": np.concat((dec1, dec2, dec3)),  # Degrees
    "band": ["r"] * num_samples,  # All red.
    "exptime": np.full(num_samples, 30.0),  # Seconds
    # Noise information is random from distributions around DP2 values.
    "pixelScale": np.full(num_samples, 0.2),  # Arcseconds per pixel
    "seeing": np.random.uniform(low=0.84, high=0.88, size=num_samples),  # arcseconds
    "skyBg": np.random.uniform(low=1420.0, high=1430.0, size=num_samples),  # ADU
    "zeroPoint": np.random.normal(
        loc=30.0, scale=0.5, size=num_samples
    ),  # magnitudes to produce 1 count (ADU)
}
obs_table = LSSTObsTable(fake_data)
survey_info = SurveyInfo(obstable=obs_table, survey_name="LSST")

We create a corresponding mdwarf flare model for the three objects at each of the three (fixed positions). We use the same samplers for the other parameters (temp, radius, etc.)

In [ ]:
model2 = MDwarfFlareModel(
    star_temp=star_temp,
    star_radius=star_radius_sampler,
    flare_fwhm=flare_fwhm_sampler,
    flare_amplitude=flare_amplitude_sampler,
    t0=t0,
    ra=GivenValueList([0.0, 180.0, 90.0], stateful=False),
    dec=GivenValueList([0.0, -30.0, 30.0], stateful=False),
    distance=100.0,  # Fixed distance in parsecs
    redshift=0.0,
    node_label="model",
)

Then we plot the resulting light curves.

In [ ]:
df = simulate_lightcurves(
    model=model2,
    num_samples=3,
    survey_info=survey_info,
    rng=np.random.default_rng(42),  # Use a fixed random seed.
)
plot_all_results(df)

As you can see, we only evaluate the model at the times it is observed. But those are not always useful. 

### Time Window Filtering

LightCurveLynx allows you to limit the evaluation to just a window around the object's `t0` parameter. You can define this restriction in either the rest frame or the observer frame.

In [ ]:
df = simulate_lightcurves(
    model=model2,
    num_samples=3,
    survey_info=survey_info,
    rng=np.random.default_rng(42),  # Use the SAME fixed random seed.
    rest_time_window_offset=(-2.0, 2.0),
)
plot_all_results(df)

### Reusing State to Partially Filter

The filtering shown in the previous cell may be more extreme than we want. For example, if we have at least one observation that occurs during the flare, we might want to extract all the other observations of that object to analyze non-flaring behavior.

We can do this by running the simulation twice. During the first run, we use `rest_time_window_offset` to filter all objects whose flares we do not observe. Then we extract the parameters for these objects (including positions and `t0` and use that to simulate those objects) for all times in our data set.

In [ ]:
# First simulation with strict filtering.
df = simulate_lightcurves(
    model=model2,
    num_samples=3,
    survey_info=survey_info,
    rest_time_window_offset=(-2.0, 2.0),
    drop_empty_rows=True,  # Drop objects we don't see at all.
    rng=np.random.default_rng(42),  # Use the SAME fixed random seed.
)

# Extract the parameters of the unfiltered objects.
states = GraphState.from_list(df["params"].to_numpy())

# Resimulate for all times. We don't set rng here, because we are replaying from state.
df = simulate_lightcurves(
    model=model2,
    graph_state=states,  # Simulate from the saved parameters
    num_samples=states.num_samples,
    survey_info=survey_info,
)
plot_all_results(df)

As you can see, we get the full light curves for the two mdwarf flares that have observations during the flare and we filter out the entire sample for the object that is not observed during the flare.